In [1]:
import os
import sys
from cno import CNO

import math
import time
import datetime
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
import torch
from torch.utils.data import Dataset
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torch.optim.lr_scheduler import CosineAnnealingLR, LambdaLR
# from YourDataset import YourDataset  # Import your custom dataset here
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler
from torchinfo import summary
import torchprofile
import matplotlib.pyplot as plt

import json, time, tempfile

from soap import SOAP

import pickle

torch.manual_seed(23)

scaler = GradScaler()

DTYPE = torch.float32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 200
plt.rcParams["font.family"] = "serif"

import scipy.stats as stats

Using device: cuda


In [2]:
# Define your custom loss function here
class CustomLoss(nn.Module):
    def __init__(self, Par):
        super(CustomLoss, self).__init__()
        self.Par = Par

    def forward(self, y_pred, y_true):
        # Implement your custom loss calculation here
        # loss = torch.mean((y_pred - y_true) ** 2)  # Example: Mean Squared Error
        y_true = (y_true - self.Par["out_shift"])/self.Par["out_scale"]
        y_pred = (y_pred - self.Par["out_shift"])/self.Par["out_scale"]
        loss = torch.norm(y_true-y_pred, p=2)/torch.norm(y_true, p=2)
        return loss

class YourDataset(Dataset):
    def __init__(self, x, y, transform=None):
        self.x = x
        self.y = y
        self.transform = transform

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        x_sample = self.x[idx]
        y_sample = self.y[idx]

        if self.transform:
            x_sample, y_sample = self.transform(x_sample, y_sample)

        return x_sample, y_sample


def preprocess(traj_i, traj_o, Par):
    x = traj_i.transpose(1,0,2,3) #sliding_window_view(traj_i[:,:,:,:], window_shape=Par['lf'], axis=1 ).transpose(0,1,4,2,3).reshape(-1,Par['lf'],Par['nx'], Par['ny'])[:, [0,-1]] # BS, 2, nx, ny
    y = traj_o.transpose(1,0,2,3) #sliding_window_view(traj_o[:,:,:,:], window_shape=Par['lf'], axis=1 ).transpose(0,1,4,2,3).reshape(-1,Par['lf'],Par['nx'], Par['ny'])            # BS, lf, nx, ny
    
    print('x: ', x.shape)
    print('y: ', y.shape)
    print()
    return x,y

In [3]:
#########################
begin_time = time.time()
traj_i = np.load(f"../data/lr8_data.npy").astype(np.float32)/256 #[nt, nx, ny]
traj_i = np.expand_dims(traj_i, axis=0) #[1, nt, nx, ny]
traj_o = np.load(f"../data/hr_data.npy").astype(np.float32)/256 #[nt, nx, ny]
traj_o = np.expand_dims(traj_o, axis=0) #[1, nt, nx, ny]

print(f"traj_i: {traj_i.shape}")
print(f"traj_o: {traj_o.shape}")

print(f"Data Loading Time: {time.time() - begin_time:.1f}s")


traj_i_train = traj_i[:, :800]
traj_i_val   = traj_i[:, 800:900]
traj_i_test  = traj_i[:, 900:]

traj_o_train = traj_o[:, :800]
traj_o_val   = traj_o[:, 800:900]
traj_o_test  = traj_o[:, 900:]

Par = {}
# Par['nt'] = 100 
Par['nx'] = traj_i_train.shape[2]
Par['ny'] = traj_i_train.shape[3]
Par['nf'] = 1
Par['d_emb'] = 128

Par['lb'] = 1
Par['lf'] = 1

Par["ld"] = 512
Par["n_channels"] = 16
Par["k"] = 3
Par["DEVICE"] = device
Par["DTYPE"] = DTYPE
Par["inp_ch"] = Par['nf']*Par['lb']
Par["out_ch"] = Par['nf']*Par['lf']

# Par['temp'] = Par['nt'] - Par['lb'] - Par['lf'] + 2

Par['num_epochs'] = 20000 #500 #50

begin_time = time.time()
print('\nTrain Dataset')
x_train, y_train = preprocess(traj_i_train, traj_o_train, Par)
print('\nValidation Dataset')
x_val, y_val  = preprocess(traj_i_val, traj_o_val, Par)
print('\nTest Dataset')
x_test, y_test  = preprocess(traj_i_test, traj_o_test, Par)
print(f"Data Preprocess Time: {time.time() - begin_time:.1f}s")

# sys.exit()


Par['inp_scale'] = np.max(x_train) - np.min(x_train)
Par['inp_shift'] = np.min(x_train)
Par['out_scale'] = np.max(y_train) - np.min(y_train)
Par['out_shift'] = np.min(y_train)

print(f"Par:\n{Par}")

# with open('Par.pkl', 'wb') as f:
#     pickle.dump(Par, f)

# sys.exit()
#########################

# Create custom datasets
x_train_tensor = torch.tensor(x_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)

x_val_tensor   = torch.tensor(x_val,   dtype=torch.float32)
y_val_tensor   = torch.tensor(y_val,   dtype=torch.float32)

x_test_tensor  = torch.tensor(x_test,  dtype=torch.float32)
y_test_tensor  = torch.tensor(y_test,  dtype=torch.float32)

train_dataset = YourDataset(x_train_tensor, y_train_tensor)
val_dataset = YourDataset(x_val_tensor, y_val_tensor)
test_dataset = YourDataset(x_test_tensor, y_test_tensor)

# Define data loaders
train_batch_size = 20
val_batch_size   = 20
test_batch_size  = 20
train_loader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=val_batch_size)
test_loader = DataLoader(test_dataset, batch_size=test_batch_size)

traj_i: (1, 1000, 128, 256)
traj_o: (1, 1000, 128, 256)
Data Loading Time: 0.2s

Train Dataset
x:  (800, 1, 128, 256)
y:  (800, 1, 128, 256)


Validation Dataset
x:  (100, 1, 128, 256)
y:  (100, 1, 128, 256)


Test Dataset
x:  (100, 1, 128, 256)
y:  (100, 1, 128, 256)

Data Preprocess Time: 0.0s
Par:
{'nx': 128, 'ny': 256, 'nf': 1, 'd_emb': 128, 'lb': 1, 'lf': 1, 'ld': 512, 'n_channels': 16, 'k': 3, 'DEVICE': device(type='cuda'), 'DTYPE': torch.float32, 'inp_ch': 1, 'out_ch': 1, 'num_epochs': 20000, 'inp_scale': 0.8432488, 'inp_shift': -0.018359842, 'out_scale': 0.79296875, 'out_shift': 0.05078125}


In [ ]:
# model = DeepOKan(Par).to(device).to(torch.float32) 

in_channels = x_train.shape[1]   # 1
out_channels = y_train.shape[1]  # 1
n_modes = (64,64)
hidden_channels = 64


# Initialize your Unet2D model
model = CNO(Par,
            in_dim=Par['inp_ch'],
            out_dim=Par['out_ch'],
            size_hw=(Par['nx'], Par['ny']),
            N_layers=4).to(device).to(torch.float32) 

model.eval()

path_model = 'models/best_model.pt'
model.load_state_dict(torch.load(path_model))

# print(summary(model, input_size=((1,)+x_train.shape[1:], (1,)) ) )

# # Adjust the dimensions as per your model's input size
# dummy_x = x_train_tensor[0:1].to(device)
# dummy_t = t_train_tensor[0:1].to(device)
# dummy_input = (dummy_x, dummy_t)

dummy_x = x_train_tensor[0:1].to(device)
print(f"dummy_x: {dummy_x.shape}")

dummy_y = model(dummy_x)
print(f"dummy_y: {dummy_y.shape}")

print(summary(model, input_size=(1,)+dummy_x.shape[1:] ))

# Profile the model
flops = 2*torchprofile.profile_macs(model, dummy_x)
print(f"GFLOPs: {(flops/10**9):.2f}")

# Define loss function and optimizer
criterion = CustomLoss(Par)

dummy_x: torch.Size([1, 1, 128, 256])
dummy_y: torch.Size([1, 1, 128, 256])
Layer (type:depth-idx)                        Output Shape              Param #
CNO                                           [1, 1, 128, 256]          --
├─LiftProjectBlock: 1-1                       [1, 13, 128, 256]         --
│    └─CNOBlock: 2-1                          [1, 64, 128, 256]         --
│    │    └─Conv2d: 3-1                       [1, 64, 128, 256]         640
│    │    └─Identity: 3-2                     [1, 64, 128, 256]         --
│    │    └─CNO_LReLu: 3-3                    [1, 64, 128, 256]         --
│    └─Conv2d: 2-2                            [1, 13, 128, 256]         7,501
├─Sequential: 1-8                             --                        (recursive)
│    └─ResNet: 2-3                            [1, 13, 128, 256]         --
│    │    └─Sequential: 3-4                   --                        6,240
├─ModuleList: 1-9                             --                        (recur

/oscar/home/voommen/apps/torch_env/lib64/python3.9/site-packages/torchprofile/profile.py:22: UserWarning: No handlers found: "aten::_upsample_bicubic2d_aa". Skipped.
  warnings.warn('No handlers found: "{}". Skipped.'.format(


# Inference Time

In [5]:
inference_time_ls = []

model.eval()

for i in range(15):
    begin_time = time.time()
    with torch.no_grad():
        y_pred = model(dummy_x.to(device))
    
    end_time = time.time()
    inference_time = end_time - begin_time
    print(f"Inference time: {inference_time:.5e}")
    inference_time_ls.append(inference_time)

print()
print(f"mean: {np.mean(inference_time_ls[5:]):.5e}")

Inference time: 3.61991e-03
Inference time: 3.06511e-03
Inference time: 2.90561e-03
Inference time: 2.95734e-03
Inference time: 2.88725e-03
Inference time: 1.28460e-02
Inference time: 1.30844e-02
Inference time: 1.30947e-02
Inference time: 1.30916e-02
Inference time: 1.30932e-02
Inference time: 1.30758e-02
Inference time: 1.30978e-02
Inference time: 1.30823e-02
Inference time: 1.30749e-02
Inference time: 1.30832e-02

mean: 1.30624e-02


# Peak VRAM

In [6]:
torch.backends.cudnn.benchmark = False  # keep runs reproducible

model.eval()


# Warmup
with torch.no_grad():
    y_pred = model(dummy_x.to(device))


torch.cuda.synchronize()

torch.cuda.reset_peak_memory_stats() 
with torch.no_grad():
    y_pred = model(dummy_x.to(device))

torch.cuda.synchronize()


# ---- Read peaks (bytes) and report in GB ----
peak_alloc_GB   = torch.cuda.max_memory_allocated()  / 1e9
peak_resvd_GB   = torch.cuda.max_memory_reserved()   / 1e9
print(f"Peak VRAM (allocated): {peak_alloc_GB:.4f} GB")
print(f"Peak VRAM (reserved) : {peak_resvd_GB:.4f} GB")
print("Config: batch=1, dtype=", DTYPE, ", device=", device)

Peak VRAM (allocated): 0.2975 GB
Peak VRAM (reserved) : 0.4907 GB
Config: batch=1, dtype= torch.float32 , device= cuda


# Sanity Check

In [10]:
# y_true_ls = []
# y_pred_ls = []

# model.eval()
# train_loss = 0.0
# with torch.no_grad():
#     for x, t, y_true in train_loader:
#         with autocast():
#             y_pred = model(x.to(device), t.to(device))
#             loss   = criterion(y_pred, y_true.to(device))
#         train_loss += loss.item()
#         y_true_ls.append(y_true.detach().cpu().numpy())
#         y_pred_ls.append(y_pred.detach().cpu().numpy())

# train_loss /= len(train_loader)
# print(f"Train Loss: {train_loss:.4e}")

# TRAIN_TRUE = np.concatenate(y_true_ls, axis=0).reshape(-1, Par['lf'], Par['nx'], Par['ny']).astype(np.float32)
# TRAIN_PRED = np.concatenate(y_pred_ls, axis=0).reshape(-1, Par['lf'], Par['nx'], Par['ny']).astype(np.float32)

# print(f"TRAIN_TRUE: {TRAIN_TRUE.shape}, DTYPE: {TRAIN_TRUE.dtype}")
# print(f"TRAIN_PRED: {TRAIN_PRED.shape}, DTYPE: {TRAIN_PRED.dtype}")



# y_true_ls = []
# y_pred_ls = []

# model.eval()
# val_loss = 0.0
# with torch.no_grad():
#     for x, t, y_true in val_loader:
#         with autocast():
#             y_pred = model(x.to(device), t.to(device))
#             loss   = criterion(y_pred, y_true.to(device))
#         val_loss += loss.item()
#         y_true_ls.append(y_true.detach().cpu().numpy())
#         y_pred_ls.append(y_pred.detach().cpu().numpy())

# val_loss /= len(val_loader)
# print(f"Val Loss: {val_loss:.4e}")

# VAL_TRUE = np.concatenate(y_true_ls, axis=0).reshape(-1, Par['lf'], Par['nx'], Par['ny']).astype(np.float32)
# VAL_PRED = np.concatenate(y_pred_ls, axis=0).reshape(-1, Par['lf'], Par['nx'], Par['ny']).astype(np.float32)

# print(f"VAL_TRUE: {VAL_TRUE.shape}, DTYPE: {VAL_TRUE.dtype}")
# print(f"VAL_PRED: {VAL_PRED.shape}, DTYPE: {VAL_PRED.dtype}")



y_true_ls = []
y_pred_ls = []

model.eval()
val_loss = 0.0
with torch.no_grad():
    for x, y_true in val_loader:
        if True:
            y_pred = model(x.to(device))
            loss = criterion(y_pred, y_true.to(device))
        val_loss += loss.item()
        y_true_ls.append(y_true.detach().cpu().numpy())
        y_pred_ls.append(y_pred.detach().cpu().numpy())

val_loss /= len(val_loader)
print(f"Val Loss: {val_loss:.4e}")


VAL_TRUE = np.concatenate(y_true_ls, axis=0).reshape(-1, Par['lf'], Par['nx'], Par['ny']).astype(np.float32)
VAL_PRED = np.concatenate(y_pred_ls, axis=0).reshape(-1, Par['lf'], Par['nx'], Par['ny']).astype(np.float32)

print(f"VAL_TRUE: {VAL_TRUE.shape}, DTYPE: {VAL_TRUE.dtype}")
print(f"VAL_PRED: {VAL_PRED.shape}, DTYPE: {VAL_PRED.dtype}")



y_true_ls = []
y_pred_ls = []

model.eval()
test_loss = 0.0
with torch.no_grad():
    for x, y_true in test_loader:
        if True:
            y_pred = model(x.to(device))
            loss = criterion(y_pred, y_true.to(device))
        test_loss += loss.item()
        y_true_ls.append(y_true.detach().cpu().numpy())
        y_pred_ls.append(y_pred.detach().cpu().numpy())

test_loss /= len(test_loader)
print(f"Test Loss: {test_loss:.4e}")


TEST_TRUE = np.concatenate(y_true_ls, axis=0).reshape(-1, Par['lf'], Par['nx'], Par['ny']).astype(np.float32)
TEST_PRED = np.concatenate(y_pred_ls, axis=0).reshape(-1, Par['lf'], Par['nx'], Par['ny']).astype(np.float32)

print(f"TEST_TRUE: {TEST_TRUE.shape}, DTYPE: {TEST_TRUE.dtype}")
print(f"TEST_PRED: {TEST_PRED.shape}, DTYPE: {TEST_PRED.dtype}")

Val Loss: 1.5117e-01
VAL_TRUE: (100, 1, 128, 256), DTYPE: float32
VAL_PRED: (100, 1, 128, 256), DTYPE: float32
Test Loss: 1.5294e-01
TEST_TRUE: (100, 1, 128, 256), DTYPE: float32
TEST_PRED: (100, 1, 128, 256), DTYPE: float32


In [11]:
np.save("VAL_TRUE.npy", VAL_TRUE)
np.save("VAL_PRED.npy", VAL_PRED)

np.save("TEST_TRUE.npy", TEST_TRUE)
np.save("TEST_PRED.npy", TEST_PRED)